In [ ]:
# install the necessary libraries
!pip install rpy2==3.5.1 xgboost --quiet

import itertools
import time
import math
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score
)
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.7/201.7 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Mounted at /content/drive


In [ ]:
# rpy2 setup
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
pandas2ri.activate()
%load_ext rpy2.ipython

set_seed = 123
np.random.seed(set_seed)

In [ ]:

################################################################################
# load R script for simulating data
################################################################################
%%R
source("drive/MyDrive/sim_dgp.R")



In [ ]:

################################################################################
# function to call R's simulate_experiment_dataset(...)
################################################################################
def generate_sim_data(
    N=100, p=3,
    regularity=0.2, missing_prop=0.1,
    num_sync_groups=1,
    process_types="ar1", use_lags=False,
    block_missing=True, block_length=1.5,
    outcome_time=10, master_seed=511
):
    """
    Calls R's simulate_experiment_dataset() with the given parameters.
    Returns two Pandas DataFrames: df_vars (long format) & df_outcomes.
    Columns in df_vars => [ID, time, variable, value]
    Columns in df_outcomes => [ID, time, Y, true_prob]
    """
    ro.r.assign("N", N)
    ro.r.assign("p", p)
    ro.r.assign("regularity", regularity)
    ro.r.assign("missing_prop", missing_prop)
    ro.r.assign("num_sync_groups", num_sync_groups)

    if isinstance(process_types, list):
        ro.r.assign("process_types", ro.StrVector(process_types))
    else:
        ro.r.assign("process_types", ro.StrVector([process_types]*p))

    ro.r.assign("use_lags", use_lags)
    ro.r.assign("block_missing", block_missing)
    ro.r.assign("block_length", block_length)
    ro.r.assign("outcome_time", outcome_time)
    ro.r.assign("master_seed", master_seed)

    ro.r('''
        sim_dataset <- simulate_experiment_dataset(
          N            = N,
          p            = p,
          freq_min     = 0.1,
          freq_max     = 50.0,
          regularity   = regularity,
          num_sync_groups = num_sync_groups,
          process_types = process_types,
          missing_prop = missing_prop,
          block_missing = block_missing,
          block_length  = block_length,
          outcome_time  = outcome_time,
          beta          = c(0, seq(-1, 1, length.out = 10)),
          use_lags      = use_lags,
          lag_hours     = ifelse(use_lags, 2, 0),
          master_seed   = master_seed
        )
    ''')

    df_vars_r     = ro.r('sim_dataset$vars_long')
    df_outcomes_r = ro.r('sim_dataset$outcome_df')

    # Convert to pandas
    df_vars     = df_vars_r
    df_outcomes = df_outcomes_r

    # dtypes
    df_vars["ID"]      = df_vars["ID"].astype(int)
    df_vars["time"]    = df_vars["time"].astype(float)
    df_outcomes["ID"]  = df_outcomes["ID"].astype(int)
    df_outcomes["time"]= df_outcomes["time"].astype(float)

    return df_vars, df_outcomes



In [ ]:

################################################################################
# bin each subject's timeseries into hour-bins up to 'outcome_time'
#    then forward-fill. Then flatten to produce  (outcome_time * p) features.
################################################################################
def bin_and_forward_fill(df_sub, feature_list, outcome_time=10):
    """
    - Bin time into half-hour intervals from 0..(2*outcome_time - 1).
    - If multiple rows occur in the same half-hour bin for a feature, average them.
    - Forward-fill missing bins.
    - Finally, flatten into shape (2 * outcome_time * p,).
    """

    p = len(feature_list)
    n_bins = 2 * outcome_time

    data_arr = np.full((n_bins, p), np.nan, dtype=float)



    df_sub["HalfHourBin"] = (df_sub["time"] // 0.5).astype(int)

    grouped = df_sub.groupby(["HalfHourBin", "variable"])["value"].mean().reset_index()


    for hh_bin in range(n_bins):
        for feat_idx, feat in enumerate(feature_list):
            row_match = grouped[
                (grouped["HalfHourBin"] == hh_bin) & (grouped["variable"] == feat)
            ]
            if len(row_match) == 1:
                data_arr[hh_bin, feat_idx] = row_match["value"].values[0]


    for feat_idx in range(p):
        for hh_bin in range(1, n_bins):
            if np.isnan(data_arr[hh_bin, feat_idx]):
                data_arr[hh_bin, feat_idx] = data_arr[hh_bin - 1, feat_idx]

        data_arr[:, feat_idx] = np.nan_to_num(data_arr[:, feat_idx], nan=0.0)


    return data_arr.flatten(order="C")


def build_binned_dataset(df_vars, df_outcomes, outcome_time=10):
    """
    For each subject => produce (2 * outcome_time * p) columns + label.
    Return a DataFrame => columns: [ID, label, <f0_h0>, <f1_h0>, ..., <f0_h1>, ...].
    """
    feature_list = sorted(df_vars["variable"].unique())
    subject_ids  = df_vars["ID"].unique()

    rows = []
    for sid in subject_ids:
        sub_df  = df_vars[df_vars["ID"] == sid].copy()
        sub_out = df_outcomes[df_outcomes["ID"] == sid]

        label_val = 0
        if len(sub_out) == 1:
            label_val = sub_out["Y"].values[0]

        data_flat = bin_and_forward_fill(sub_df, feature_list, outcome_time=outcome_time)

        row_dict = {"ID": sid, "label": label_val}
        idx = 0

        n_bins = 2 * outcome_time

        for hh_bin in range(n_bins):
            for feat_i, feat_name in enumerate(feature_list):
                col_name = f"{feat_name}_h{hh_bin}"
                row_dict[col_name] = data_flat[idx]
                idx += 1

        rows.append(row_dict)

    df_out = pd.DataFrame(rows)
    return df_out


In [ ]:

################################################################################
# run_single_scenario => does entire pipeline for "LogisticRegression" & "XGB"
################################################################################
def run_single_scenario(
    N, p,
    regularity,
    missing_prop,
    num_sync_groups,
    process_types,
    use_lags,
    block_missing=True,
    block_length=1.5,
    outcome_time=10,
    master_seed=123
):
    """
    1) Generate data in R, 2) bin+flatten each subject => (outcome_time * p) features,
    3) 70/15/15 split => train/val/test (though val isn't strictly used in LR/XGB),
    4) Fit LR & XGB, 5) Evaluate on test => gather metrics.

    We'll return TWO entries in a list, one for each model, each a dict of metrics:
      e.g. [
        { 'model':'LR', 'auc':..., 'accuracy':..., etc. },
        { 'model':'XGB', ... }
      ]
    """
    # generate data
    df_vars, df_outcomes = generate_sim_data(
        N=N,
        p=p,
        regularity=regularity,
        missing_prop=missing_prop,
        num_sync_groups=num_sync_groups,
        process_types=process_types,
        use_lags=use_lags,
        block_missing=block_missing,
        block_length=block_length,
        outcome_time=outcome_time,
        master_seed=master_seed
    )

    # compute ground-truth stats
    mean_true_prob = df_outcomes["true_prob"].mean()
    prop_positive  = df_outcomes["Y"].mean()

    # build binned dataset
    df_binned = build_binned_dataset(df_vars, df_outcomes, outcome_time=outcome_time)

    # 70/15/15 split
    df_binned = df_binned.sample(frac=1.0, random_state=42).reset_index(drop=True)
    N_total = len(df_binned)
    n_train = int(0.70*N_total)
    n_val   = int(0.15*N_total)
    n_test  = N_total - n_train - n_val

    train_df = df_binned.iloc[:n_train].copy()
    val_df   = df_binned.iloc[n_train : n_train + n_val].copy()
    test_df  = df_binned.iloc[n_train + n_val : ].copy()

    X_cols = [c for c in df_binned.columns if c not in ["ID","label"]]

    X_train = train_df[X_cols].values
    y_train = train_df["label"].values

    # not strictly used:
    X_val   = val_df[X_cols].values
    y_val   = val_df["label"].values

    X_test  = test_df[X_cols].values
    y_test  = test_df["label"].values

    results_list = []

    # Fit Logistic Regression
    lr_start = time.time()
    lr_model = LogisticRegression(max_iter=2000, random_state=42)
    lr_model.fit(X_train, y_train)
    lr_time = time.time() - lr_start

    y_pred_lr = lr_model.predict(X_test)
    if hasattr(lr_model,"predict_proba"):
        y_prob_lr = lr_model.predict_proba(X_test)[:,1]
        auc_lr    = roc_auc_score(y_test, y_prob_lr)
    else:
        auc_lr    = float('nan')

    acc_lr     = accuracy_score(y_test, y_pred_lr)
    prec_lr    = precision_score(y_test, y_pred_lr, zero_division=0)
    rec_lr     = recall_score(y_test, y_pred_lr, zero_division=0)
    f1_lr      = f1_score(y_test, y_pred_lr, zero_division=0)

    results_list.append({
        "model":         "LR",
        "test_loss":     float('nan'),
        "auc":           auc_lr,
        "accuracy":      acc_lr,
        "precision":     prec_lr,
        "recall":        rec_lr,
        "f1":            f1_lr,
        "runtime_sec":   lr_time,
        "mean_true_prob": float(mean_true_prob),
        "prop_positive":  float(prop_positive),
    })

    #  Fit XGB
    xgb_start = time.time()
    xgb = XGBClassifier(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss',
        early_stopping_rounds=10
    )

    eval_set = [(X_val, y_val)]
    xgb.fit(X_train, y_train, eval_set=eval_set, verbose=False)
    xgb_time = time.time() - xgb_start

    y_pred_xgb = xgb.predict(X_test)
    y_prob_xgb = xgb.predict_proba(X_test)[:,1]


    auc_xgb   = roc_auc_score(y_test, y_prob_xgb)
    acc_xgb   = accuracy_score(y_test, y_pred_xgb)
    prec_xgb  = precision_score(y_test, y_pred_xgb, zero_division=0)
    rec_xgb   = recall_score(y_test, y_pred_xgb, zero_division=0)
    f1_xgb    = f1_score(y_test, y_pred_xgb, zero_division=0)

    results_list.append({
        "model":         "XGB",
        "test_loss":     float('nan'),
        "auc":           auc_xgb,
        "accuracy":      acc_xgb,
        "precision":     prec_xgb,
        "recall":        rec_xgb,
        "f1":            f1_xgb,
        "runtime_sec":   xgb_time,
        "mean_true_prob": float(mean_true_prob),
        "prop_positive":  float(prop_positive),
    })

    return results_list



In [ ]:

################################################################################
# Full Factorial Loop + Collect Results
################################################################################
import math

Ns = [200, 1000]
regularities = [0.2, 0.8]
missing_props = [0.1, 0.4]
sync_pattern_values = [1, "p"]
process_type_values = ["homogeneous", "mixed"]
outcome_dep_values = ["direct", "lagged"]

scenario_grid = list(itertools.product(
    Ns,
    regularities,
    missing_props,
    sync_pattern_values,
    process_type_values,
    outcome_dep_values
))

all_results = []
overall_start = time.time()

for scenario in scenario_grid:
    (N,
     regularity,
     missing_prop,
     sync_pattern,
     process_type_flag,
     outcome_dep) = scenario

    # parse process_type
    p = 10
    if process_type_flag=="homogeneous":
        process_types = "ar1"
    else:
        repeats = math.ceil(p/3)
        big_array = (["ar1","rw","seasonal"]*repeats)[:p]
        process_types = big_array

    # parse sync
    if sync_pattern=="p":
        num_sync_groups = max(1, int(round(math.sqrt(p))))
    else:
        num_sync_groups = 1

    # parse outcome dependency
    use_lags = (outcome_dep=="lagged")

    print("===============================================================")
    print(f"Scenario => N={N}, reg={regularity}, missing={missing_prop}, "
          f"sync={sync_pattern}, process={process_type_flag}, outcome={outcome_dep}")
    print("===============================================================")

    start_scenario = time.time()
    results_for_models = run_single_scenario(
        N=N,
        p=p,
        regularity=regularity,
        missing_prop=missing_prop,
        num_sync_groups=num_sync_groups,
        process_types=process_types,
        use_lags=use_lags,
        block_missing=True,
        block_length=1.5,
        outcome_time=10,
        master_seed=123
    )
    scenario_time = time.time() - start_scenario


    for model_result in results_for_models:
        row = {
            "N":              N,
            "regularity":     regularity,
            "missing_prop":   missing_prop,
            "sync_pattern":   sync_pattern,
            "process_type":   process_type_flag,
            "outcome_dep":    outcome_dep,
            "model":          model_result["model"],
            "test_loss":      model_result["test_loss"],
            "auc":            model_result["auc"],
            "accuracy":       model_result["accuracy"],
            "precision":      model_result["precision"],
            "recall":         model_result["recall"],
            "f1":             model_result["f1"],
            "runtime_sec":    model_result["runtime_sec"],
            "mean_true_prob": model_result["mean_true_prob"],
            "prop_positive":  model_result["prop_positive"],
        }
        all_results.append(row)

overall_time = time.time() - overall_start
print(f"\nAll scenarios completed in {overall_time/60:.2f} minutes.\n")

df_results = pd.DataFrame(all_results)
df_results.sort_values(by=["N","regularity","missing_prop","model"], inplace=True)
print(df_results.head(30))

df_results.to_csv("drive/MyDrive/lr_xgb_factorial_results_long.csv", index=False)
print("\nResults saved to lr_xgb_factorial_results_long.csv")


Scenario => N=200, reg=0.2, missing=0.1, sync=1, process=homogeneous, outcome=direct
Scenario => N=200, reg=0.2, missing=0.1, sync=1, process=homogeneous, outcome=lagged
Scenario => N=200, reg=0.2, missing=0.1, sync=1, process=mixed, outcome=direct
Scenario => N=200, reg=0.2, missing=0.1, sync=1, process=mixed, outcome=lagged
Scenario => N=200, reg=0.2, missing=0.1, sync=p, process=homogeneous, outcome=direct
Scenario => N=200, reg=0.2, missing=0.1, sync=p, process=homogeneous, outcome=lagged
Scenario => N=200, reg=0.2, missing=0.1, sync=p, process=mixed, outcome=direct
Scenario => N=200, reg=0.2, missing=0.1, sync=p, process=mixed, outcome=lagged
Scenario => N=200, reg=0.2, missing=0.4, sync=1, process=homogeneous, outcome=direct
Scenario => N=200, reg=0.2, missing=0.4, sync=1, process=homogeneous, outcome=lagged
Scenario => N=200, reg=0.2, missing=0.4, sync=1, process=mixed, outcome=direct
Scenario => N=200, reg=0.2, missing=0.4, sync=1, process=mixed, outcome=lagged
Scenario => N=20